In [ ]:
!find . -name "__pycache__" -exec rm -rf {} + 2>/dev/null; echo "caches cleared"

In [ ]:
# Step 1: Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install dependencies and clone repo
import os
os.environ['MAX_JOBS'] = '2'

# Cache gsplat CUDA build on Google Drive (survives session restarts)
CACHE_DIR = '/content/drive/MyDrive/GaussianFractalLOD/.cache/torch_extensions'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['TORCH_EXTENSIONS_DIR'] = CACHE_DIR

!pip install gsplat -q
!pip install -q torchmetrics lpips tensorboard pyyaml

if not os.path.exists('/content/GaussianFractalLOD'):
    !git clone https://github.com/dedoubleyou1/GaussianFractalLOD.git

%cd /content/GaussianFractalLOD

# Always pull latest and clean caches before installing
!git pull origin main
!apt-get install -y git-lfs -qq
!git lfs install
!git lfs pull
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; echo 'caches cleared'
!pip install -e . --no-deps -q

print('Setup complete!')

In [ ]:
# NeRF Synthetic data is included in the repo (via Git LFS)
# All 8 scenes: chair, drums, ficus, hotdog, lego, materials, mic, ship
import os
REPO_DIR = os.getcwd()  # Should be /content/GaussianFractalLOD
DATA_DIR = os.path.join(REPO_DIR, 'nerf_synthetic')
assert os.path.exists(DATA_DIR), f'Data not found at {DATA_DIR}'
!ls {DATA_DIR}/

In [ ]:
from gaussianfractallod.config import Config

SCENE = "lego"

cfg = Config(
    data_dir=f"{DATA_DIR}/{SCENE}",
    num_roots=1,
    sh_degree=0,
    max_levels=9,
    # split_grad_threshold=0.00005,  # 4× lower than default
    checkpoint_dir=f"/content/drive/MyDrive/GaussianFractalLOD/checkpoints/{SCENE}_9lv2",
)
print(f"Training {SCENE}: {cfg.num_roots} root, SH{cfg.sh_degree}, {cfg.max_levels} levels")
print(f"Iterations per level: {[cfg.level_base_iterations * 2**i for i in range(cfg.max_levels)]}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from gaussianfractallod.train import train

RESUME_FROM = None  # e.g., f"{cfg.checkpoint_dir}/phase2_level_3.pt"

roots, tree = train(cfg, resume_from=RESUME_FROM)
print(f"Training complete! {tree.depth} levels")

In [ ]:
import torch
from gaussianfractallod.data import NerfSyntheticDataset
from gaussianfractallod.eval import evaluate

device = torch.device("cuda")
test_dataset = NerfSyntheticDataset(cfg.data_dir, split="test")
background = torch.tensor(cfg.background_color, device=device)

results = {}
for depth in range(tree.depth):
    r = evaluate(tree.to(device), test_dataset, depth, device, background)
    results[depth] = r
    print(f"Depth {depth}: PSNR={r['psnr']:.2f}, {r['num_gaussians']} Gaussians")

In [ ]:
import torch
import matplotlib.pyplot as plt
from gaussianfractallod.render import render_gaussians
from gaussianfractallod.data import NerfSyntheticDataset

device = torch.device("cuda")
test_dataset = NerfSyntheticDataset(cfg.data_dir, split="test")
background = torch.tensor(cfg.background_color, device=device)

gt_image, camera = test_dataset[0]
cam = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in camera.items()}

depths = list(range(tree.depth))
ncols = len(depths) + 1
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4), dpi=150)

axes[0].imshow(gt_image.numpy().clip(0, 1))
axes[0].set_title("Ground Truth", fontsize=10)
axes[0].axis("off")

for i, depth in enumerate(depths):
    with torch.no_grad():
        gaussians = tree.get_gaussians_at_depth(depth)
        rendered = render_gaussians(
            gaussians, cam["viewmat"], cam["K"],
            cam["width"], cam["height"], background,
        )
    axes[i + 1].imshow(rendered.cpu().numpy().clip(0, 1))
    axes[i + 1].set_title(f"L{depth}: {gaussians.num_gaussians:,} G", fontsize=10)
    axes[i + 1].axis("off")

plt.suptitle(f"LoD Progression — {SCENE} (SH{cfg.sh_degree})", fontsize=12)
plt.tight_layout()
plt.savefig(f"{cfg.checkpoint_dir}/lod_progression.png", dpi=150, bbox_inches="tight")
plt.show()

# PSNR vs depth curve
if results:
    fig2, ax2 = plt.subplots(1, 1, figsize=(8, 5), dpi=150)
    ds = sorted(results.keys())
    psnrs = [results[d]['psnr'] for d in ds]
    counts = [results[d]['num_gaussians'] for d in ds]
    ax2.plot(ds, psnrs, 'o-', color='steelblue', linewidth=2)
    ax2.set_xlabel('Level')
    ax2.set_ylabel('PSNR (dB)')
    ax2.set_title(f'PSNR vs Level — {SCENE}')
    ax2.grid(True, alpha=0.3)
    for d, p, c in zip(ds, psnrs, counts):
        ax2.annotate(f'{c:,}', (d, p), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8, color='gray')
    plt.tight_layout()
    plt.savefig(f"{cfg.checkpoint_dir}/psnr_vs_depth.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
import torch
from gaussianfractallod.export_ply import export_ply

device = torch.device("cuda")
export_dir = f"/content/drive/MyDrive/GaussianFractalLOD/exports/{SCENE}"

for depth in range(tree.depth):
    with torch.no_grad():
        gaussians = tree.get_gaussians_at_depth(depth)
    export_ply(gaussians, f"{export_dir}/level_{depth}.ply", sh_degree=cfg.sh_degree)

print(f"Exported to: {export_dir}")
print("Drag PLY files into https://antimatter15.com/splat/ to view in 3D")

In [ ]:
import torch
with torch.no_grad():
    g = tree.get_gaussians_at_depth(5)
    cov = g.covariance()
    eigvals = torch.linalg.eigvalsh(cov)
    ratios = eigvals[:, 2] / (eigvals[:, 0] + 1e-8)  # max/min eigenvalue
    print(f"Aspect ratios — mean: {ratios.mean():.1f}, max: {ratios.max():.1f}, median: {ratios.median():.1f}")
    print(f"Min eigenvalue: {eigvals[:, 0].min():.2e}")
    print(f"Max eigenvalue: {eigvals[:, 2].max():.2e}")

In [ ]:
import math
s = math.sqrt(1 - 2/math.pi)  # ≈ 0.603
lam = math.sqrt(2/math.pi)     # ≈ 0.798
print(f"Child 1σ scale: {s**3:.3f}")
print(f"Sibling separation: {2*lam:.3f}")
print(f"Coverage: {s**3/lam:.1%}")

In [ ]:
from gaussianfractallod.subdivide import SPREAD_FACTOR, subdivide_to_8
import inspect

print(f"SPREAD_FACTOR = {SPREAD_FACTOR}")
src = inspect.getsource(subdivide_to_8)
print("HAS NEW FORMULA" if "0.932" in src else "OLD FORMULA")

In [ ]:
import torch

print(f"{'Level':>6s} {'Gaussians':>10s} {'Avg scale':>10s} {'Min scale':>10s} {'Max scale':>10s} {'Avg aspect':>12s}")
for depth in range(tree.depth):
    with torch.no_grad():
        g = tree.get_gaussians_at_depth(depth)
        cov = g.covariance()
        eigvals = torch.linalg.eigvalsh(cov)
        scales = eigvals.sqrt()  # (N, 3) sorted ascending
        avg_scale = scales.mean(dim=-1).mean().item()
        min_scale = scales[:, 0].mean().item()  # avg of smallest axis
        max_scale = scales[:, 2].mean().item()  # avg of largest axis
        aspect = (scales[:, 2] / (scales[:, 0] + 1e-8)).mean().item()
        print(f"{depth:>6d} {g.num_gaussians:>10d} {avg_scale:>10.4f} {min_scale:>10.4f} {max_scale:>10.4f} {aspect:>12.1f}")

In [ ]:
from gaussianfractallod.train import _get_level_scale
# Should print 0.75 for level 8 with max_levels=9 (600/800)
print(_get_level_scale(8, 9))
# Should print clean steps, not powers of 2
for l in range(10):
    print(f"Level {l}: scale={_get_level_scale(l, 9):.3f}, res={int(_get_level_scale(l, 9)*800)}")

In [ ]:
# Check training convergence per level from tensorboard logs
# Or simpler: retrain and track loss at checkpoints

import torch
from gaussianfractallod.render import render_gaussians
from gaussianfractallod.loss import rendering_loss
from gaussianfractallod.data import NerfSyntheticDataset

device = torch.device("cuda")
background = torch.tensor([1.0, 1.0, 1.0], device=device)

# Sample 10 training views for quick loss estimate per level
dataset = NerfSyntheticDataset(cfg.data_dir, split="train")
sample_indices = list(range(0, 100, 10))  # 10 evenly spaced views

print(f"{'Level':>6s} {'Gaussians':>10s} {'Avg Loss':>10s} {'PSNR est':>10s}")
for depth in range(tree.depth):
    with torch.no_grad():
        g = tree.get_gaussians_at_depth(depth)
        total_loss = 0
        for idx in sample_indices:
            gt, cam = dataset[idx]
            gt = gt.to(device)
            cam = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in cam.items()}
            rendered = render_gaussians(g, cam["viewmat"], cam["K"], cam["width"], cam["height"], background)
            total_loss += rendering_loss(rendered, gt, ssim_weight=0.2).item()
        avg_loss = total_loss / len(sample_indices)
        print(f"{depth:>6d} {g.num_gaussians:>10d} {avg_loss:>10.4f} {results.get(depth, {}).get('psnr', 0):>10.2f}")

In [ ]:
import glob
ckpts = sorted(glob.glob("/content/drive/MyDrive/GaussianFractalLOD/checkpoints/lego_9levels/phase2_level_*.pt"))
for c in ckpts:
    print(c)

In [ ]:
import glob
# Check all checkpoint dirs
for path in glob.glob("/content/drive/MyDrive/GaussianFractalLOD/checkpoints/*/"):
    files = glob.glob(path + "*.pt")
    print(f"{path}: {len(files)} files")
    for f in files[:3]:
        print(f"  {f}")